# 01 Preprocess Junyi (C1 Adaptive Curriculum Engine)

Turns the raw Junyi Academy answer log into clean, reproducible training data for BKT, the GAT readiness model and the DKT/GKT baselines.

**Decisions (state these in the report):**

| Decision | Choice | Why |
|---|---|---|
| What counts as correct | First attempt correct **and** no hint used | Closer to real mastery than "eventually correct" |
| Skill unit | One skill per exercise | Nodes match the prerequisite graph, so BKT, GAT and explanations line up |
| Minimum history | Students with at least 10 answers | Stated in proposal section 5.2 |
| Split | By **student**: 20% held-out test, 5 folds on the other 80% | No student appears in both training and test |
| Long histories | Cut into chunks of 200 answers for DKT/GKT | Memory and speed; BKT uses the full history |
| Sample (PP1) | Fixed random sample of students, seed 42 | Reproducible; full data is run after PP1 |

**Leakage rules for later notebooks:** BKT parameters are fitted on training folds only, and the mastery used as a GAT feature at time *t* is computed from answers before *t* only.

Outputs go to Google Drive so a disconnect never loses them. Nothing here is committed to git.

In [ ]:
# Setup: install, mount Drive, download Junyi (about 1.2 GB, only on first run).
import sys
!{sys.executable} -m pip install -q EduData pyarrow

from pathlib import Path

try:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = Path("/content/drive/MyDrive/adaptlearn-c1")
except ImportError:  # running locally
    BASE = Path("./data")

RAW_DIR = BASE / "raw"
OUT_DIR = BASE / "junyi_processed"
RAW_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

if not (RAW_DIR / "junyi" / "junyi_ProblemLog_original.csv").exists():
    from EduData import get_data
    get_data("junyi", str(RAW_DIR))

JUNYI = RAW_DIR / "junyi"
print(sorted(p.name for p in JUNYI.glob("*.csv")))

In [ ]:
# Configuration. Change SAMPLE_STUDENTS to None for the full dataset after PP1.
SEED = 42
MIN_INTERACTIONS = 10
MAX_SEQ_LEN = 200
SAMPLE_STUDENTS = 30_000
TEST_FRACTION = 0.20
N_FOLDS = 5
CORRECT_RULE = "first_try_no_hint"   # or "first_try"

## 1. Inspect the answer log before loading it

In [ ]:
import os
import pandas as pd
import numpy as np

LOG = JUNYI / "junyi_ProblemLog_original.csv"
print(f"{os.path.getsize(LOG) / 1e9:.2f} GB")
head = pd.read_csv(LOG, nrows=5)
print(head.columns.tolist())
head

In [ ]:
# Fail loudly if the columns are not what this notebook expects.
REQUIRED = {"user_id", "exercise", "time_done", "correct"}
OPTIONAL = {"hint_used", "count_hints", "count_attempts"}
missing = REQUIRED - set(head.columns)
assert not missing, f"Answer log is missing required columns: {missing}"
USECOLS = sorted(REQUIRED | (OPTIONAL & set(head.columns)))
print("Loading columns:", USECOLS)

## 2. Load only the needed columns, in chunks, with small types

In [ ]:
def to_bool(series: pd.Series) -> pd.Series:
    text = series.astype(str).str.strip().str.lower()
    return text.isin({"true", "t", "1", "yes"})

chunks = []
for chunk in pd.read_csv(LOG, usecols=USECOLS, chunksize=2_000_000, low_memory=False):
    chunk = chunk.dropna(subset=list(REQUIRED))
    out = pd.DataFrame({
        "user_id": chunk["user_id"].astype("int64"),
        "exercise": chunk["exercise"].astype("string"),
        "time_done": pd.to_numeric(chunk["time_done"], errors="coerce"),
        "correct_raw": to_bool(chunk["correct"]),
    })
    out["hint_used"] = to_bool(chunk["hint_used"]) if "hint_used" in chunk else False
    if "count_hints" in chunk:
        out["hint_used"] |= pd.to_numeric(chunk["count_hints"], errors="coerce").fillna(0) > 0
    chunks.append(out)

log = pd.concat(chunks, ignore_index=True)
del chunks
funnel = [("raw rows (required fields present)", len(log), log.user_id.nunique())]
log.info(memory_usage="deep")

## 3. Clean

In [ ]:
ex = pd.read_csv(JUNYI / "junyi_Exercise_table.csv", low_memory=False)
live = set(ex.loc[ex["live"] == True, "name"])

log = log.dropna(subset=["time_done"])
log = log.drop_duplicates(subset=["user_id", "exercise", "time_done"])
funnel.append(("after dropping bad timestamps and duplicates", len(log), log.user_id.nunique()))

log = log[log["exercise"].isin(live)]
funnel.append(("after keeping live exercises in the exercise table", len(log), log.user_id.nunique()))

In [ ]:
# Label: what "correct" means for every model in this project.
if CORRECT_RULE == "first_try_no_hint":
    log["correct"] = (log["correct_raw"] & ~log["hint_used"]).astype("int8")
elif CORRECT_RULE == "first_try":
    log["correct"] = log["correct_raw"].astype("int8")
else:
    raise ValueError(CORRECT_RULE)
log = log.drop(columns=["correct_raw"])
print("correct rate:", round(log["correct"].mean(), 4))

In [ ]:
# Order matters: every model learns from the sequence of answers.
log = log.sort_values(["user_id", "time_done"], kind="mergesort").reset_index(drop=True)

counts = log.groupby("user_id").size()
keep = counts[counts >= MIN_INTERACTIONS].index
log = log[log["user_id"].isin(keep)]
funnel.append((f"after dropping students with < {MIN_INTERACTIONS} answers", len(log), log.user_id.nunique()))

## 4. Sample students (not rows), so every kept history stays whole

In [ ]:
rng = np.random.default_rng(SEED)
students = np.sort(log["user_id"].unique())

if SAMPLE_STUDENTS and SAMPLE_STUDENTS < len(students):
    # Stratify by history length so the sample keeps short and long learners.
    lengths = log.groupby("user_id").size().reindex(students)
    bands = pd.qcut(lengths.rank(method="first"), 4, labels=False)
    chosen = []
    for band in range(4):
        pool = students[bands.values == band]
        take = round(SAMPLE_STUDENTS * len(pool) / len(students))
        chosen.extend(rng.choice(pool, size=min(take, len(pool)), replace=False))
    students = np.sort(np.array(chosen))
    log = log[log["user_id"].isin(students)]
    funnel.append((f"sample of {len(students)} students (seed {SEED})", len(log), log.user_id.nunique()))

## 5. Index exercises and cut long histories

In [ ]:
# Exercise index follows the sorted live exercise names, so it matches the graph notebook.
exercise_index = {name: i for i, name in enumerate(sorted(live))}
log["exercise_idx"] = log["exercise"].map(exercise_index).astype("int32")

# Position within each student's history, and a chunk id for DKT/GKT (BKT ignores chunks).
log["step"] = log.groupby("user_id").cumcount().astype("int32")
log["seq_chunk"] = (log["step"] // MAX_SEQ_LEN).astype("int32")
log = log.reset_index(drop=True)

## 6. Split by student: held-out test plus 5 folds

In [ ]:
shuffled = rng.permutation(np.sort(log["user_id"].unique()))
n_test = int(round(len(shuffled) * TEST_FRACTION))
test_students = np.sort(shuffled[:n_test])
train_students = shuffled[n_test:]

fold_of = {int(u): i % N_FOLDS for i, u in enumerate(train_students)}
splits = {
    "seed": SEED,
    "test": [int(u) for u in test_students],
    "folds": {str(k): sorted(int(u) for u, f in fold_of.items() if f == k) for k in range(N_FOLDS)},
}

# Leakage check: no student in more than one place.
all_fold = [u for k in splits["folds"].values() for u in k]
assert len(all_fold) == len(set(all_fold)), "a student is in two folds"
assert not set(all_fold) & set(splits["test"]), "a test student is also in training"
print({"test": len(splits["test"]), **{f"fold_{k}": len(v) for k, v in splits["folds"].items()}})

## 7. Dataset statistics (goes in the report and PP1 slides)

In [ ]:
lengths = log.groupby("user_id").size()
stats = pd.Series({
    "students": log["user_id"].nunique(),
    "answers": len(log),
    "exercises with answers": log["exercise"].nunique(),
    "live exercises in graph": len(live),
    "mean answers per student": round(lengths.mean(), 1),
    "median answers per student": int(lengths.median()),
    "correct rate": round(log["correct"].mean(), 4),
    "answers with a hint": round(log["hint_used"].mean(), 4),
})
funnel_table = pd.DataFrame(funnel, columns=["step", "rows", "students"])
display(funnel_table)
stats

## 8. Save outputs and a manifest

In [ ]:
import hashlib, json, platform
from datetime import datetime, timezone

files = {
    "interactions.parquet": lambda p: log[["user_id", "exercise", "exercise_idx", "time_done",
                                           "step", "seq_chunk", "correct", "hint_used"]].to_parquet(p, index=False),
    "splits.json": lambda p: p.write_text(json.dumps(splits)),
    "exercise_index.json": lambda p: p.write_text(json.dumps(exercise_index)),
    "stats.json": lambda p: p.write_text(stats.to_json()),
    "funnel.json": lambda p: p.write_text(funnel_table.to_json(orient="records")),
}
checksums = {}
for name, write in files.items():
    path = OUT_DIR / name
    write(path)
    checksums[name] = hashlib.sha256(path.read_bytes()).hexdigest()

manifest = {
    "created_at": datetime.now(timezone.utc).isoformat(),
    "config": {"SEED": SEED, "MIN_INTERACTIONS": MIN_INTERACTIONS, "MAX_SEQ_LEN": MAX_SEQ_LEN,
               "SAMPLE_STUDENTS": SAMPLE_STUDENTS, "TEST_FRACTION": TEST_FRACTION,
               "N_FOLDS": N_FOLDS, "CORRECT_RULE": CORRECT_RULE},
    "versions": {"python": platform.python_version(), "pandas": pd.__version__, "numpy": np.__version__},
    "sha256": checksums,
}
(OUT_DIR / "manifest.json").write_text(json.dumps(manifest, indent=2))
print(json.dumps(manifest, indent=2))

## Next

- `02_graph_junyi.ipynb`: build the prerequisite graph, check direction against the expert ratings, rater agreement, Neo4j import for visuals.
- `03_bkt_baseline.ipynb`: pyBKT fitted on the training folds, AUC and accuracy on the held-out test.

Before committing this notebook: **Edit > Clear all outputs**.